# adapter_registry\nGenerated from 00_common/adapter_registry.py for Databricks notebook import.\n

In [ ]:
"""Plugin architecture for extensible ingestion adapters."""\n\nfrom __future__ import annotations\n\nimport importlib\nimport logging\nfrom abc import ABC, abstractmethod\nfrom typing import Any, Optional\n\nlogger = logging.getLogger(__name__)\n\n\nclass IngestAdapter(ABC):\n    """\n    Abstract base class for all ingestion adapters.\n    \n    Subclasses must implement the ingest() method.\n    """\n    \n    def __init__(self, source: dict[str, Any], source_options: Optional[dict] = None):\n        """\n        Initialize adapter.\n        \n        Args:\n            source: Source configuration from metadata\n            source_options: Source-specific options\n        """\n        self.source = source\n        self.source_options = source_options or {}\n    \n    @abstractmethod\n    def ingest(self, spark=None) -> Any:\n        """\n        Ingest data from source.\n        \n        Args:\n            spark: Spark session (optional)\n            \n        Returns:\n            PySpark DataFrame or iterator of records\n        """\n        pass\n    \n    def validate(self) -> tuple[bool, list[str]]:\n        """\n        Validate adapter configuration.\n        \n        Returns:\n            (is_valid, error_messages)\n        """\n        return True, []\n\n\nclass AdapterRegistry:\n    """\n    Central registry for managing ingestion adapters.\n    \n    Allows dynamic discovery and instantiation of adapters.\n    """\n    \n    _adapters: dict[str, type[IngestAdapter]] = {}\n    \n    @classmethod\n    def register(cls, source_type: str, adapter_class: type[IngestAdapter]) -> None:\n        """Register an adapter."""\n        cls._adapters[source_type.upper()] = adapter_class\n        logger.info(f"Registered adapter: {source_type}")\n    \n    @classmethod\n    def get_adapter(cls, source_type: str) -> Optional[type[IngestAdapter]]:\n        """Get an adapter by source type."""\n        return cls._adapters.get(source_type.upper())\n    \n    @classmethod\n    def list_adapters(cls) -> list[str]:\n        """List all registered adapters."""\n        return list(cls._adapters.keys())\n    \n    @classmethod\n    def get_or_create_adapter(\n        cls,\n        source_type: str,\n        source: dict,\n        source_options: Optional[dict] = None,\n    ) -> Optional[IngestAdapter]:\n        """Get or instantiate an adapter."""\n        adapter_class = cls.get_adapter(source_type)\n        if not adapter_class:\n            logger.warning(f"No adapter found for source_type: {source_type}")\n            return None\n        return adapter_class(source=source, source_options=source_options)\n\n\ndef auto_discover_adapters(adapter_module: str = "notebooks.01_ingestion.adapters") -> None:\n    """\n    Auto-discover and register adapters from a module.\n    \n    Args:\n        adapter_module: Module path to search for adapters\n    """\n    try:\n        module = importlib.import_module(adapter_module)\n        for item_name in dir(module):\n            item =getattr(module, item_name)\n            if (\n                isinstance(item, type)\n                and issubclass(item, IngestAdapter)\n                and item is not IngestAdapter\n                and hasattr(item, "SOURCE_TYPE")\n            ):\n                source_type = getattr(item, "SOURCE_TYPE")\n                AdapterRegistry.register(source_type, item)\n                logger.info(f"Auto-discovered adapter: {source_type}")\n    except Exception as e:\n        logger.warning(f"Failed to auto-discover adapters: {e}")\n